# Strong model comparison / ensemble on v2 feature spaces

In [1]:
# =========================
# 0. Configuration
# =========================

from pathlib import Path
import os
import gc
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.utils.validation import check_is_fitted

try:
    from scipy.stats import rankdata
except Exception:
    rankdata = None

try:
    import lightgbm as lgb
    HAS_LGB = True
except Exception as e:
    HAS_LGB = False
    print("LightGBM unavailable:", repr(e))

try:
    import xgboost as xgb
    HAS_XGB = True
except Exception as e:
    HAS_XGB = False
    print("XGBoost unavailable:", repr(e))

try:
    from catboost import CatBoostClassifier, Pool
    HAS_CATBOOST = True
except Exception as e:
    HAS_CATBOOST = False
    print("CatBoost unavailable:", repr(e))

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    HAS_OPTUNA = True
except Exception as e:
    HAS_OPTUNA = False
    print("Optuna unavailable:", repr(e))

RANDOM_STATE = 42
N_SPLITS = 5
N_JOBS = max(1, (os.cpu_count() or 8) - 1)

# Heavy toggles.
RUN_OPTUNA_LGBM = True          # Если долго, поставь False.
OPTUNA_N_TRIALS = 70            # 50-100 обычно достаточно.
OPTUNA_TIMEOUT = 2 * 60 * 60    # seconds; Optuna остановится раньше по trials или timeout.

RUN_CATBOOST = True
RUN_XGBOOST = True
RUN_EXTRA_TREES = True
RUN_LINEAR_MODELS = True
RUN_HIST_GB = True

# Full-fit test predictions usually help a little, but cost extra time.
# CV-average predictions are safer; full-fit predictions are saved separately and also used in some blends.
RETRAIN_FULL_FOR_SUBMISSION = True

# Blend search.
RANDOM_BLEND_TRIALS = 20000
BLEND_TOP_N_MODELS = 12

# Output dirs.
SUBMISSION_DIRNAME = "submissions_v2"
MODEL_OUTPUT_DIRNAME = "model_outputs_v2"

np.random.seed(RANDOM_STATE)
print("N_JOBS:", N_JOBS)
print("HAS_LGB:", HAS_LGB, "HAS_XGB:", HAS_XGB, "HAS_CATBOOST:", HAS_CATBOOST, "HAS_OPTUNA:", HAS_OPTUNA)

N_JOBS: 13
HAS_LGB: True HAS_XGB: True HAS_CATBOOST: True HAS_OPTUNA: True


## 1. 

In [2]:
# =========================
# 1. Locate repository root
# =========================

def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "prepared_data").exists():
            return p
    # fallback: current working directory
    return start

ROOT = find_repo_root()
PREP = ROOT / "prepared_data"
SUB_DIR = ROOT / SUBMISSION_DIRNAME
OUT_DIR = ROOT / MODEL_OUTPUT_DIRNAME

SUB_DIR.mkdir(exist_ok=True, parents=True)
OUT_DIR.mkdir(exist_ok=True, parents=True)

print("ROOT:", ROOT)
print("PREP:", PREP)
print("SUB_DIR:", SUB_DIR)
print("OUT_DIR:", OUT_DIR)

assert PREP.exists(), f"prepared_data not found at {PREP}"

ROOT: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton
PREP: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/prepared_data
SUB_DIR: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/submissions_v2
OUT_DIR: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/model_outputs_v2


In [3]:
# =========================
# 2. Load prepared files
# =========================

def read_csv_maybe(path: Path, **kwargs):
    if path.exists():
        return pd.read_csv(path, **kwargs)
    return None

def load_parquet_required(path: Path) -> pd.DataFrame:
    assert path.exists(), f"Missing required file: {path}"
    print("Loading:", path.name)
    df = pd.read_parquet(path)
    # стабилизируем память
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].astype("float32")
        elif pd.api.types.is_integer_dtype(df[c]):
            # частоты/флаги оставим компактными, но LightGBM всё равно нормально съест
            pass
    print(path.name, df.shape)
    return df

# Targets / ids
y_path = PREP / "y_train.csv"
train_idx_path = PREP / "train_index.csv"
test_idx_path = PREP / "test_index.csv"

assert y_path.exists(), "prepared_data/y_train.csv is required"
y_df = pd.read_csv(y_path)

if "target" in y_df.columns:
    y = y_df["target"].astype(int).values
else:
    # fallback: one-column csv
    y = y_df.iloc[:, -1].astype(int).values

if train_idx_path.exists():
    train_index = pd.read_csv(train_idx_path).iloc[:, 0].values
else:
    train_index = np.arange(len(y))

if test_idx_path.exists():
    test_index = pd.read_csv(test_idx_path).iloc[:, 0].values
else:
    test_index = None

print("y:", y.shape, "positive rate:", y.mean(), "positives:", int(y.sum()))

# Matrices
X_raw_train = load_parquet_required(PREP / "X_train_raw_clean.parquet")
X_raw_test = load_parquet_required(PREP / "X_test_raw_clean.parquet")

# Prefer v2 engineered if exists
eng_train_path = PREP / "X_train_engineered_v2.parquet"
eng_test_path = PREP / "X_test_engineered_v2.parquet"
if not eng_train_path.exists():
    eng_train_path = PREP / "X_train_engineered.parquet"
    eng_test_path = PREP / "X_test_engineered.parquet"

X_eng_train = load_parquet_required(eng_train_path)
X_eng_test = load_parquet_required(eng_test_path)

# Optional model meta features
meta_train_path = PREP / "X_train_model_meta_features.parquet"
meta_test_path = PREP / "X_test_model_meta_features.parquet"
if meta_train_path.exists() and meta_test_path.exists():
    X_meta_train = load_parquet_required(meta_train_path)
    X_meta_test = load_parquet_required(meta_test_path)
else:
    X_meta_train = pd.DataFrame(index=X_raw_train.index)
    X_meta_test = pd.DataFrame(index=X_raw_test.index)
    print("No model_meta features found.")

# If test_index absent, infer from test matrix index
if test_index is None:
    test_index = np.asarray(X_raw_test.index)

assert len(X_raw_train) == len(y)
assert len(X_raw_test) == len(test_index)

y: (247972,) positive rate: 0.013493458938912458 positives: 3346
Loading: X_train_raw_clean.parquet
X_train_raw_clean.parquet (247972, 1360)
Loading: X_test_raw_clean.parquet
X_test_raw_clean.parquet (106274, 1360)
Loading: X_train_engineered_v2.parquet
X_train_engineered_v2.parquet (247972, 4089)
Loading: X_test_engineered_v2.parquet
X_test_engineered_v2.parquet (106274, 4089)
Loading: X_train_model_meta_features.parquet
X_train_model_meta_features.parquet (247972, 4)
Loading: X_test_model_meta_features.parquet
X_test_model_meta_features.parquet (106274, 4)


## 2. Собираем единый feature pool

`feature_sets.json` хранит списки колонок. Чтобы любые feature set'ы нормально собирались, делаем общий пул:

`raw_clean` + `engineered_v2` + `model_meta`

Дубликаты колонок удаляем в пользу первого появления.

In [4]:
# =========================
# 3. Build feature pool
# =========================

def concat_unique(dfs):
    out = pd.concat(dfs, axis=1)
    out = out.loc[:, ~out.columns.duplicated()]
    return out

X_pool_train = concat_unique([X_raw_train, X_eng_train, X_meta_train])
X_pool_test = concat_unique([X_raw_test, X_eng_test, X_meta_test])

# Align columns just in case
common_cols = [c for c in X_pool_train.columns if c in X_pool_test.columns]
X_pool_train = X_pool_train[common_cols]
X_pool_test = X_pool_test[common_cols]

# Replace rare infs just in case
X_pool_train = X_pool_train.replace([np.inf, -np.inf], 0)
X_pool_test = X_pool_test.replace([np.inf, -np.inf], 0)

print("X_pool_train:", X_pool_train.shape)
print("X_pool_test:", X_pool_test.shape)
print("Missing train:", int(X_pool_train.isna().sum().sum()))
print("Missing test:", int(X_pool_test.isna().sum().sum()))

X_pool_train: (247972, 5449)
X_pool_test: (106274, 5449)
Missing train: 0
Missing test: 0


In [5]:
# =========================
# 4. Load feature sets and benchmark
# =========================

feature_sets_path = PREP / "feature_sets.json"
assert feature_sets_path.exists(), "prepared_data/feature_sets.json is required"

with open(feature_sets_path, "r") as f:
    feature_sets = json.load(f)

# Keep only existing columns
feature_sets = {
    name: [c for c in cols if c in X_pool_train.columns and c in X_pool_test.columns]
    for name, cols in feature_sets.items()
}
feature_sets = {name: cols for name, cols in feature_sets.items() if len(cols) > 0}

bench_path = PREP / "feature_set_benchmark_lgb_v2.csv"
if bench_path.exists():
    benchmark = pd.read_csv(bench_path).sort_values("mean_auc", ascending=False)
    display(benchmark)
else:
    benchmark = pd.DataFrame({"feature_set": list(feature_sets), "mean_auc": np.nan})
    print("No benchmark file found; using available feature sets.")

print("Available feature sets:", len(feature_sets))
print(pd.DataFrame({"feature_set": list(feature_sets), "n_features": [len(v) for v in feature_sets.values()]}).sort_values("n_features").to_string(index=False))

,feature_set,n_features,mean_auc,std_auc,min_auc,max_auc,mean_best_iter
0,v2_peak_freq_te,1361,0.704447,0.007076,0.696042,0.713817,278.8
1,v2_pca_svd_stack,1165,0.702001,0.007667,0.689298,0.712599,136.8
2,top500_plus_engineered,4589,0.700776,0.006085,0.689698,0.707336,144.8
3,v2_linear_stack_compact,1415,0.700512,0.007603,0.687477,0.710530,119.4
4,v2_nodrift_top700_engineered,1869,0.700050,0.007462,0.686818,0.709171,200.0
5,v2_lgb_top500_meta,1515,0.699564,0.007325,0.687798,0.709421,122.6
6,v2_full_beauty,5449,0.698456,0.005415,0.689406,0.704980,171.2
7,top700_plus_engineered,4789,0.698379,0.010816,0.684117,0.713455,224.6
8,v2_stable_top700_engineered,1869,0.698172,0.008060,0.687225,0.710002,136.8
9,v2_lgb_top700_meta_interactions,4195,0.696840,0.011274,0.677955,0.707236,148.8


Available feature sets: 24
                    feature_set  n_features
                  top300_ranked         300
                  top500_ranked         500
           stable_top500_ranked         500
                  top700_ranked         700
           stable_top700_ranked         700
          nodrift_top700_ranked         700
                   v2_meta_only         750
                 top1000_ranked        1000
               v2_pca_svd_stack        1165
                      raw_clean        1360
                v2_peak_freq_te        1361
        v2_linear_stack_compact        1415
             v2_lgb_top500_meta        1515
            raw_clean_plus_meta        1670
    v2_stable_top700_engineered        1869
   v2_nodrift_top700_engineered        1869
          v2_interactions_heavy        3991
v2_lgb_top700_meta_interactions        4195
            v2_model_meta_heavy        4345
         top500_plus_engineered        4589
         top700_plus_engineered        4789
     

## 3. Какие feature sets берём

По твоему прогону лучший быстрый benchmark:

- `v2_peak_freq_te` — около `0.7044`;
- `v2_pca_svd_stack` — около `0.7020`;
- `top500_plus_engineered` — около `0.7008`;
- `v2_linear_stack_compact` — около `0.7005`;
- `v2_nodrift_top700_engineered` — около `0.7000`.

Это **кратно лучше старых 0.66–0.67 на локальной валидации**. Но в модели берём не только топ-1: для ансамбля важна диверсификация.

In [6]:
# =========================
# 5. Select feature sets for modeling
# =========================

# Preferred by observed v2 benchmark.
PREFERRED_FEATURE_SETS = [
    "v2_peak_freq_te",
    "v2_pca_svd_stack",
    "top500_plus_engineered",
    "v2_linear_stack_compact",
    "v2_nodrift_top700_engineered",
    "v2_lgb_top500_meta",
    "v2_full_beauty",
    "top700_plus_engineered",
    "v2_stable_top700_engineered",
]

selected_feature_sets = [fs for fs in PREFERRED_FEATURE_SETS if fs in feature_sets]

# Add top benchmark rows if new names appear.
if bench_path.exists():
    for fs in benchmark["feature_set"].head(10).tolist():
        if fs in feature_sets and fs not in selected_feature_sets:
            selected_feature_sets.append(fs)

print("Selected feature sets:")
for fs in selected_feature_sets:
    row = benchmark[benchmark["feature_set"] == fs] if bench_path.exists() else pd.DataFrame()
    auc = row["mean_auc"].iloc[0] if len(row) and "mean_auc" in row else np.nan
    print(f"  {fs:35s} n={len(feature_sets[fs]):5d} benchmark_auc={auc}")

Selected feature sets:
  v2_peak_freq_te                     n= 1361 benchmark_auc=0.7044473079172959
  v2_pca_svd_stack                    n= 1165 benchmark_auc=0.7020006209565551
  top500_plus_engineered              n= 4589 benchmark_auc=0.7007759147256556
  v2_linear_stack_compact             n= 1415 benchmark_auc=0.7005124749428541
  v2_nodrift_top700_engineered        n= 1869 benchmark_auc=0.7000498539054123
  v2_lgb_top500_meta                  n= 1515 benchmark_auc=0.6995640348475874
  v2_full_beauty                      n= 5449 benchmark_auc=0.6984560437310235
  top700_plus_engineered              n= 4789 benchmark_auc=0.698379176250694
  v2_stable_top700_engineered         n= 1869 benchmark_auc=0.6981718756167898
  v2_lgb_top700_meta_interactions     n= 4195 benchmark_auc=0.696839694044045


## 4. Group-aware CV

Из-за дублей в train/test нельзя делать наивный split. Если одинаковые feature-vector'ы попадут одновременно в train и validation, локальная метрика станет завышенной.

Поэтому группы считаются по hash от `raw_clean`-признаков. Одинаковые строки получают один group id и попадают в один fold.

In [7]:
# =========================
# 6. Group-aware folds
# =========================

print("Computing full-vector hash groups from raw_clean...")
t0 = time.time()
groups_hash = pd.util.hash_pandas_object(X_raw_train, index=False).values
groups, group_uniques = pd.factorize(groups_hash, sort=False)
print("Groups:", len(np.unique(groups)), "for rows:", len(groups), "time:", round(time.time() - t0, 1), "s")

cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X_raw_train, y, groups=groups))

for fold, (_, va_idx) in enumerate(folds):
    print(
        f"fold={fold}",
        "size=", len(va_idx),
        "pos=", int(y[va_idx].sum()),
        "pos_rate=", round(float(y[va_idx].mean()), 5),
        "groups=", len(np.unique(groups[va_idx]))
    )

Computing full-vector hash groups from raw_clean...
Groups: 236365 for rows: 247972 time: 1.6 s
fold=0 size= 49596 pos= 670 pos_rate= 0.01351 groups= 47269
fold=1 size= 49594 pos= 669 pos_rate= 0.01349 groups= 47272
fold=2 size= 49594 pos= 669 pos_rate= 0.01349 groups= 47271
fold=3 size= 49594 pos= 669 pos_rate= 0.01349 groups= 47274
fold=4 size= 49594 pos= 669 pos_rate= 0.01349 groups= 47279


## 5. Утилиты для сохранения и blend

Все модели сохраняют:

- OOF prediction: `model_outputs_v2/oof_<model>.csv`;
- test prediction: `model_outputs_v2/test_pred_<model>.csv`;
- submission: `submissions_v2/submission_<model>.csv`.

Для ROC-AUC важен порядок объектов, поэтому кроме probability-blend делаем ещё rank-blend.

In [8]:
# =========================
# 7. Saving / blending utilities
# =========================

def safe_name(s: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in s)

def make_submission(score, name: str):
    score = np.asarray(score, dtype=float)
    sub = pd.DataFrame({"index": test_index, "score": score})
    out_path = SUB_DIR / f"submission_{safe_name(name)}.csv"
    sub.to_csv(out_path, index=False)
    print("Saved submission:", out_path)
    return out_path

def save_pred_files(oof, test_pred, name: str):
    oof_df = pd.DataFrame({"index": train_index, "target": y, "score": np.asarray(oof, dtype=float)})
    test_df = pd.DataFrame({"index": test_index, "score": np.asarray(test_pred, dtype=float)})
    oof_path = OUT_DIR / f"oof_{safe_name(name)}.csv"
    test_path = OUT_DIR / f"test_pred_{safe_name(name)}.csv"
    oof_df.to_csv(oof_path, index=False)
    test_df.to_csv(test_path, index=False)
    print("Saved:", oof_path)
    print("Saved:", test_path)

def auc_score(y_true, pred):
    return float(roc_auc_score(y_true, pred))

def rank01(x):
    x = np.asarray(x, dtype=float)
    if rankdata is not None:
        r = rankdata(x, method="average")
    else:
        r = pd.Series(x).rank(method="average").values
    return (r - r.min()) / max(1e-12, (r.max() - r.min()))

def rank_blend(pred_matrix, weights=None):
    arr = np.asarray(pred_matrix, dtype=float)
    if weights is None:
        weights = np.ones(arr.shape[1]) / arr.shape[1]
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    ranked = np.column_stack([rank01(arr[:, i]) for i in range(arr.shape[1])])
    return ranked @ weights

def prob_blend(pred_matrix, weights=None):
    arr = np.asarray(pred_matrix, dtype=float)
    if weights is None:
        weights = np.ones(arr.shape[1]) / arr.shape[1]
    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()
    return arr @ weights

model_oof = {}
model_test = {}
model_meta = []

## 6. Sample weights для label-noise экспериментов

В v2-пайплайне появился `noise_candidates_by_oof.csv`. Это не гарантированно ошибочные строки, а **подозрительные** строки, где модель уверенно не согласна с разметкой.

Для нескольких моделей попробуем не удаление, а мягкий downweight. Это безопаснее: редкие реальные паттерны не выкидываются полностью.

In [9]:
# =========================
# 8. Optional noise-aware sample weights
# =========================

base_sample_weight = np.ones(len(y), dtype="float32")
noise_path = PREP / "noise_candidates_by_oof.csv"

if noise_path.exists():
    noise_df = pd.read_csv(noise_path)
    # Map by train index if possible
    idx_to_pos = {idx: pos for pos, idx in enumerate(train_index)}
    if "index" in noise_df.columns:
        pos = noise_df["index"].map(idx_to_pos).dropna().astype(int).values
    else:
        pos = noise_df.index.values

    # Prefer top 0.5% flag if available.
    if "is_top_005_noise" in noise_df.columns:
        flagged = noise_df.loc[noise_df["is_top_005_noise"].astype(int) == 1, "index"] if "index" in noise_df.columns else noise_df.index[noise_df["is_top_005_noise"].astype(int) == 1]
        if "index" in noise_df.columns:
            flagged_pos = pd.Series(flagged).map(idx_to_pos).dropna().astype(int).values
        else:
            flagged_pos = np.asarray(flagged, dtype=int)
    else:
        # fallback: top 0.5% by suspicious_score
        k = max(1, int(0.005 * len(y)))
        top = noise_df.sort_values("suspicious_score", ascending=False).head(k)
        if "index" in top.columns:
            flagged_pos = top["index"].map(idx_to_pos).dropna().astype(int).values
        else:
            flagged_pos = top.index.values

    base_sample_weight[flagged_pos] = 0.55
    print("Noise-aware weights: downweighted rows:", len(flagged_pos))
else:
    print("No noise_candidates_by_oof.csv found; using unit weights.")

Noise-aware weights: downweighted rows: 1240


## 7. LightGBM: основной рабочий класс моделей

LightGBM — главный кандидат: он быстрее всего, хорошо работает на табличных sparse/high-dimensional данных и уже дал лучшие результаты.

Здесь несколько конфигураций, чтобы модели были разнообразными:

- `regularized`: аккуратная регуляризация;
- `deeper`: более глубокие взаимодействия;
- `dart`: стохастический вариант;
- `goss`: другой способ сэмплирования градиентов;
- `noisy_downweight`: та же идея, но с пониженным весом подозрительных строк;
- `optuna_best`: если включён Optuna.

In [10]:
# =========================
# 9. LightGBM training helpers
# =========================

assert HAS_LGB, "LightGBM is required for this notebook."

pos = float(y.sum())
neg = float(len(y) - y.sum())
scale_pos_weight = neg / max(pos, 1.0)
print("scale_pos_weight:", scale_pos_weight)

def lgb_base_params(seed=42):
    return {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "learning_rate": 0.018,
        "num_leaves": 48,
        "max_depth": -1,
        "min_data_in_leaf": 120,
        "feature_fraction": 0.82,
        "bagging_fraction": 0.82,
        "bagging_freq": 1,
        "lambda_l1": 0.05,
        "lambda_l2": 18.0,
        "min_gain_to_split": 0.0,
        "scale_pos_weight": scale_pos_weight,
        "verbosity": -1,
        "seed": seed,
        "feature_fraction_seed": seed,
        "bagging_seed": seed,
        "data_random_seed": seed,
        "num_threads": N_JOBS,
        "force_col_wise": True,
    }

def train_lgb_cv(
    name: str,
    feature_set_name: str,
    params: dict,
    num_boost_round: int = 6000,
    early_stopping_rounds: int = 250,
    sample_weight=None,
    retrain_full: bool = RETRAIN_FULL_FOR_SUBMISSION,
):
    cols = feature_sets[feature_set_name]
    print("\n" + "=" * 100)
    print("LightGBM model:", name)
    print("feature_set:", feature_set_name, "n_features:", len(cols))
    print("=" * 100)

    X = X_pool_train[cols]
    X_test = X_pool_test[cols]

    oof = np.zeros(len(y), dtype="float32")
    test_pred = np.zeros(len(X_test), dtype="float64")
    best_iters = []
    fold_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(folds):
        print(f"\n{name} fold {fold}")

        X_tr = X.iloc[tr_idx]
        X_va = X.iloc[va_idx]
        y_tr = y[tr_idx]
        y_va = y[va_idx]

        w_tr = sample_weight[tr_idx] if sample_weight is not None else None
        w_va = sample_weight[va_idx] if sample_weight is not None else None

        dtrain = lgb.Dataset(X_tr, label=y_tr, weight=w_tr, free_raw_data=False)
        dvalid = lgb.Dataset(X_va, label=y_va, weight=w_va, reference=dtrain, free_raw_data=False)

        fold_params = params.copy()
        fold_params["seed"] = params.get("seed", RANDOM_STATE) + fold
        fold_params["feature_fraction_seed"] = fold_params["seed"]
        fold_params["bagging_seed"] = fold_params["seed"]
        fold_params["data_random_seed"] = fold_params["seed"]

        model = lgb.train(
            fold_params,
            dtrain,
            valid_sets=[dvalid],
            valid_names=["valid"],
            num_boost_round=num_boost_round,
            callbacks=[
                lgb.early_stopping(early_stopping_rounds, verbose=False),
                lgb.log_evaluation(300),
            ],
        )

        pred_va = model.predict(X_va, num_iteration=model.best_iteration)
        pred_test = model.predict(X_test, num_iteration=model.best_iteration)

        oof[va_idx] = pred_va.astype("float32")
        test_pred += pred_test / N_SPLITS
        best_iters.append(int(model.best_iteration))
        auc = auc_score(y_va, pred_va)
        fold_aucs.append(auc)
        print(f"{name} fold={fold} auc={auc:.6f} best_iter={model.best_iteration}")

        del dtrain, dvalid, model, X_tr, X_va
        gc.collect()

    mean_auc = auc_score(y, oof)
    print(f"\n{name} OOF AUC={mean_auc:.6f}")
    print("fold aucs:", [round(a, 6) for a in fold_aucs])
    print("best_iters:", best_iters, "mean:", np.mean(best_iters))

    save_pred_files(oof, test_pred, name)
    make_submission(test_pred, name)

    model_oof[name] = oof
    model_test[name] = test_pred
    model_meta.append({
        "model": name,
        "family": "lgbm",
        "feature_set": feature_set_name,
        "n_features": len(cols),
        "oof_auc": mean_auc,
        "fold_auc_mean": float(np.mean(fold_aucs)),
        "fold_auc_std": float(np.std(fold_aucs)),
        "mean_best_iter": float(np.mean(best_iters)),
        "cv_test_pred": True,
        "fullfit_test_pred": False,
    })

    # Full retrain prediction with average best iteration.
    if retrain_full:
        full_rounds = int(np.ceil(np.mean(best_iters) * 1.12))
        full_rounds = max(80, full_rounds)
        print(f"\nRetraining full {name} for {full_rounds} rounds...")

        full_params = params.copy()
        full_params["seed"] = params.get("seed", RANDOM_STATE) + 1000
        full_params["feature_fraction_seed"] = full_params["seed"]
        full_params["bagging_seed"] = full_params["seed"]
        full_params["data_random_seed"] = full_params["seed"]

        w_full = sample_weight if sample_weight is not None else None
        dfull = lgb.Dataset(X, label=y, weight=w_full, free_raw_data=False)
        full_model = lgb.train(
            full_params,
            dfull,
            num_boost_round=full_rounds,
            callbacks=[lgb.log_evaluation(300)],
        )
        full_test_pred = full_model.predict(X_test, num_iteration=full_rounds)
        full_name = name + "_fullfit"
        # No OOF for fullfit, so don't include in OOF blending. Save test submission only.
        pd.DataFrame({"index": test_index, "score": full_test_pred}).to_csv(OUT_DIR / f"test_pred_{safe_name(full_name)}.csv", index=False)
        make_submission(full_test_pred, full_name)
        model_meta.append({
            "model": full_name,
            "family": "lgbm_fullfit",
            "feature_set": feature_set_name,
            "n_features": len(cols),
            "oof_auc": np.nan,
            "fold_auc_mean": np.nan,
            "fold_auc_std": np.nan,
            "mean_best_iter": full_rounds,
            "cv_test_pred": False,
            "fullfit_test_pred": True,
        })

        del dfull, full_model
        gc.collect()

    return oof, test_pred

scale_pos_weight: 73.10998206814106


In [11]:
# =========================
# 10. Optional Optuna tuning for LightGBM
# =========================

best_optuna_params = None
best_optuna_feature_set = selected_feature_sets[0] if selected_feature_sets else "v2_peak_freq_te"

if RUN_OPTUNA_LGBM and HAS_OPTUNA and HAS_LGB and best_optuna_feature_set in feature_sets:
    print("Running Optuna on feature set:", best_optuna_feature_set)

    opt_cols = feature_sets[best_optuna_feature_set]
    X_opt = X_pool_train[opt_cols]

    tr_idx, va_idx = folds[0]
    X_tr, X_va = X_opt.iloc[tr_idx], X_opt.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    def objective(trial):
        params = lgb_base_params(seed=RANDOM_STATE)
        params.update({
            "learning_rate": trial.suggest_float("learning_rate", 0.008, 0.045, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 24, 160),
            "max_depth": trial.suggest_categorical("max_depth", [-1, 5, 6, 7, 8, 10, 12]),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 40, 500),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.55, 0.95),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.55, 0.95),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-4, 8.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1.0, 120.0, log=True),
            "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 0.08),
            "extra_trees": trial.suggest_categorical("extra_trees", [False, True]),
        })

        dtrain = lgb.Dataset(X_tr, label=y_tr, free_raw_data=False)
        dvalid = lgb.Dataset(X_va, label=y_va, reference=dtrain, free_raw_data=False)
        model = lgb.train(
            params,
            dtrain,
            valid_sets=[dvalid],
            valid_names=["valid"],
            num_boost_round=5000,
            callbacks=[
                lgb.early_stopping(180, verbose=False),
                lgb.log_evaluation(0),
            ],
        )
        pred = model.predict(X_va, num_iteration=model.best_iteration)
        score = auc_score(y_va, pred)
        return score

    study = optuna.create_study(direction="maximize", study_name="lgbm_v2_feature_space")
    study.optimize(objective, n_trials=OPTUNA_N_TRIALS, timeout=OPTUNA_TIMEOUT, show_progress_bar=True)

    print("Optuna best value:", study.best_value)
    print("Optuna best params:", study.best_params)

    best_optuna_params = lgb_base_params(seed=2026)
    best_optuna_params.update(study.best_params)

    pd.DataFrame([{"best_value": study.best_value, **study.best_params}]).to_csv(OUT_DIR / "optuna_lgbm_best_params.csv", index=False)

    del X_opt, X_tr, X_va
    gc.collect()
else:
    print("Skipping Optuna.")

Running Optuna on feature set: v2_peak_freq_te


  0%|          | 0/70 [00:00<?, ?it/s]

Optuna best value: 0.7160792784229122
Optuna best params: {'learning_rate': 0.009014718831186133, 'num_leaves': 78, 'max_depth': 5, 'min_data_in_leaf': 135, 'feature_fraction': 0.949225912871495, 'bagging_fraction': 0.866610261982951, 'bagging_freq': 4, 'lambda_l1': 6.731891723491176, 'lambda_l2': 1.0076191990400498, 'min_gain_to_split': 0.01532045424908467, 'extra_trees': True}


In [12]:
# =========================
# 11. Train LightGBM model zoo
# =========================

lgb_specs = []

if "v2_peak_freq_te" in feature_sets:
    params = lgb_base_params(seed=101)
    params.update({
        "learning_rate": 0.014,
        "num_leaves": 64,
        "min_data_in_leaf": 90,
        "feature_fraction": 0.78,
        "bagging_fraction": 0.82,
        "lambda_l1": 0.02,
        "lambda_l2": 22.0,
    })
    lgb_specs.append(("lgb_peak_regularized", "v2_peak_freq_te", params, None))

if "v2_peak_freq_te" in feature_sets:
    params = lgb_base_params(seed=102)
    params.update({
        "learning_rate": 0.010,
        "num_leaves": 96,
        "max_depth": 9,
        "min_data_in_leaf": 70,
        "feature_fraction": 0.70,
        "bagging_fraction": 0.80,
        "lambda_l1": 0.08,
        "lambda_l2": 35.0,
    })
    lgb_specs.append(("lgb_peak_deeper", "v2_peak_freq_te", params, None))

if "v2_pca_svd_stack" in feature_sets:
    params = lgb_base_params(seed=103)
    params.update({
        "learning_rate": 0.018,
        "num_leaves": 40,
        "max_depth": 7,
        "min_data_in_leaf": 130,
        "feature_fraction": 0.85,
        "bagging_fraction": 0.88,
        "lambda_l1": 0.3,
        "lambda_l2": 18.0,
    })
    lgb_specs.append(("lgb_pca_svd_stack", "v2_pca_svd_stack", params, None))

if "top500_plus_engineered" in feature_sets:
    params = lgb_base_params(seed=104)
    params.update({
        "learning_rate": 0.014,
        "num_leaves": 48,
        "min_data_in_leaf": 150,
        "feature_fraction": 0.60,
        "bagging_fraction": 0.82,
        "lambda_l1": 0.08,
        "lambda_l2": 55.0,
    })
    lgb_specs.append(("lgb_top500_engineered", "top500_plus_engineered", params, None))

if "v2_linear_stack_compact" in feature_sets:
    params = lgb_base_params(seed=105)
    params.update({
        "learning_rate": 0.020,
        "num_leaves": 32,
        "max_depth": 6,
        "min_data_in_leaf": 110,
        "feature_fraction": 0.90,
        "bagging_fraction": 0.88,
        "lambda_l1": 0.4,
        "lambda_l2": 15.0,
    })
    lgb_specs.append(("lgb_linear_stack_compact", "v2_linear_stack_compact", params, None))

if "v2_nodrift_top700_engineered" in feature_sets:
    params = lgb_base_params(seed=106)
    params.update({
        "learning_rate": 0.012,
        "num_leaves": 56,
        "min_data_in_leaf": 140,
        "feature_fraction": 0.72,
        "bagging_fraction": 0.82,
        "lambda_l1": 0.1,
        "lambda_l2": 45.0,
    })
    lgb_specs.append(("lgb_nodrift_top700", "v2_nodrift_top700_engineered", params, None))

if "v2_lgb_top500_meta" in feature_sets:
    params = lgb_base_params(seed=107)
    params.update({
        "learning_rate": 0.016,
        "num_leaves": 44,
        "max_depth": 7,
        "min_data_in_leaf": 95,
        "feature_fraction": 0.86,
        "bagging_fraction": 0.78,
        "lambda_l1": 0.15,
        "lambda_l2": 28.0,
    })
    lgb_specs.append(("lgb_top500_meta", "v2_lgb_top500_meta", params, None))

# DART can help diversity but is usually slower.
if "v2_peak_freq_te" in feature_sets:
    params = lgb_base_params(seed=108)
    params.update({
        "boosting_type": "dart",
        "learning_rate": 0.018,
        "num_leaves": 48,
        "min_data_in_leaf": 120,
        "feature_fraction": 0.78,
        "bagging_fraction": 0.82,
        "drop_rate": 0.08,
        "skip_drop": 0.55,
        "lambda_l1": 0.08,
        "lambda_l2": 30.0,
    })
    lgb_specs.append(("lgb_peak_dart", "v2_peak_freq_te", params, None))

# GOSS: different sampling, useful for diversity.
if "v2_peak_freq_te" in feature_sets:
    params = lgb_base_params(seed=109)
    params.update({
        "boosting_type": "goss",
        "learning_rate": 0.014,
        "num_leaves": 52,
        "min_data_in_leaf": 100,
        "feature_fraction": 0.82,
        "lambda_l1": 0.02,
        "lambda_l2": 25.0,
    })
    # bagging_fraction/freq are incompatible with goss
    params.pop("bagging_fraction", None)
    params.pop("bagging_freq", None)
    lgb_specs.append(("lgb_peak_goss", "v2_peak_freq_te", params, None))

# Noise-aware version
if "v2_peak_freq_te" in feature_sets and noise_path.exists():
    params = lgb_base_params(seed=110)
    params.update({
        "learning_rate": 0.014,
        "num_leaves": 64,
        "min_data_in_leaf": 90,
        "feature_fraction": 0.78,
        "bagging_fraction": 0.82,
        "lambda_l1": 0.04,
        "lambda_l2": 28.0,
    })
    lgb_specs.append(("lgb_peak_noise_downweight", "v2_peak_freq_te", params, base_sample_weight))

# Optuna-tuned version
if best_optuna_params is not None and best_optuna_feature_set in feature_sets:
    lgb_specs.append(("lgb_optuna_best", best_optuna_feature_set, best_optuna_params, None))

print("LGB specs:")
for name, fs, params, sw in lgb_specs:
    print(name, "->", fs, "weights:", sw is not None)

for name, fs, params, sw in lgb_specs:
    train_lgb_cv(name, fs, params, sample_weight=sw)

LGB specs:
lgb_peak_regularized -> v2_peak_freq_te weights: False
lgb_peak_deeper -> v2_peak_freq_te weights: False
lgb_pca_svd_stack -> v2_pca_svd_stack weights: False
lgb_top500_engineered -> top500_plus_engineered weights: False
lgb_linear_stack_compact -> v2_linear_stack_compact weights: False
lgb_nodrift_top700 -> v2_nodrift_top700_engineered weights: False
lgb_top500_meta -> v2_lgb_top500_meta weights: False
lgb_peak_dart -> v2_peak_freq_te weights: False
lgb_peak_goss -> v2_peak_freq_te weights: False
lgb_peak_noise_downweight -> v2_peak_freq_te weights: True
lgb_optuna_best -> v2_peak_freq_te weights: False

LightGBM model: lgb_peak_regularized
feature_set: v2_peak_freq_te n_features: 1361

lgb_peak_regularized fold 0
[300]	valid's auc: 0.696315
[600]	valid's auc: 0.696904
lgb_peak_regularized fold=0 auc=0.698197 best_iter=447

lgb_peak_regularized fold 1
[300]	valid's auc: 0.6956
[600]	valid's auc: 0.697368
lgb_peak_regularized fold=1 auc=0.698541 best_iter=509

lgb_peak_regul

## 8. Дополнительные модели

Здесь нужны не столько “лучшие одиночные модели”, сколько разнообразие для бленда. Даже если CatBoost / XGBoost чуть хуже LightGBM, они могут иначе ранжировать часть объектов и дать прирост в ансамбле.

In [13]:
# =========================
# 12. Generic sklearn-like CV trainer
# =========================

def train_sklearn_cv(name, feature_set_name, make_model_fn, needs_scaling=False, retrain_full=RETRAIN_FULL_FOR_SUBMISSION):
    cols = feature_sets[feature_set_name]
    print("\n" + "=" * 100)
    print("Sklearn-like model:", name)
    print("feature_set:", feature_set_name, "n_features:", len(cols))
    print("=" * 100)

    X = X_pool_train[cols]
    X_test = X_pool_test[cols]

    oof = np.zeros(len(y), dtype="float32")
    test_pred = np.zeros(len(X_test), dtype="float64")
    fold_aucs = []

    for fold, (tr_idx, va_idx) in enumerate(folds):
        print(f"\n{name} fold {fold}")
        model = make_model_fn(fold)
        model.fit(X.iloc[tr_idx], y[tr_idx])

        if hasattr(model, "predict_proba"):
            pred_va = model.predict_proba(X.iloc[va_idx])[:, 1]
            pred_test = model.predict_proba(X_test)[:, 1]
        else:
            pred_va = model.decision_function(X.iloc[va_idx])
            pred_test = model.decision_function(X_test)
            pred_va = rank01(pred_va)
            pred_test = rank01(pred_test)

        oof[va_idx] = pred_va.astype("float32")
        test_pred += pred_test / N_SPLITS
        auc = auc_score(y[va_idx], pred_va)
        fold_aucs.append(auc)
        print(f"{name} fold={fold} auc={auc:.6f}")

        del model
        gc.collect()

    mean_auc = auc_score(y, oof)
    print(f"\n{name} OOF AUC={mean_auc:.6f}")
    print("fold aucs:", [round(a, 6) for a in fold_aucs])

    save_pred_files(oof, test_pred, name)
    make_submission(test_pred, name)

    model_oof[name] = oof
    model_test[name] = test_pred
    model_meta.append({
        "model": name,
        "family": "sklearn",
        "feature_set": feature_set_name,
        "n_features": len(cols),
        "oof_auc": mean_auc,
        "fold_auc_mean": float(np.mean(fold_aucs)),
        "fold_auc_std": float(np.std(fold_aucs)),
        "mean_best_iter": np.nan,
        "cv_test_pred": True,
        "fullfit_test_pred": False,
    })

    if retrain_full:
        print(f"\nRetraining full {name}...")
        full_model = make_model_fn(999)
        full_model.fit(X, y)
        if hasattr(full_model, "predict_proba"):
            full_test_pred = full_model.predict_proba(X_test)[:, 1]
        else:
            full_test_pred = rank01(full_model.decision_function(X_test))
        full_name = name + "_fullfit"
        pd.DataFrame({"index": test_index, "score": full_test_pred}).to_csv(OUT_DIR / f"test_pred_{safe_name(full_name)}.csv", index=False)
        make_submission(full_test_pred, full_name)
        del full_model
        gc.collect()

    return oof, test_pred

In [15]:
# =========================
# 14. XGBoost
# =========================

def train_xgb_cv(name, feature_set_name, params):
    if not HAS_XGB or not RUN_XGBOOST:
        print("Skipping XGBoost.")
        return None, None

    cols = feature_sets[feature_set_name]
    print("\n" + "=" * 100)
    print("XGBoost model:", name)
    print("feature_set:", feature_set_name, "n_features:", len(cols))
    print("=" * 100)

    X = X_pool_train[cols]
    X_test = X_pool_test[cols]

    oof = np.zeros(len(y), dtype="float32")
    test_pred = np.zeros(len(X_test), dtype="float64")
    fold_aucs = []
    best_iters = []

    for fold, (tr_idx, va_idx) in enumerate(folds):
        print(f"\n{name} fold {fold}")
        model = xgb.XGBClassifier(
            **params,
            random_state=RANDOM_STATE + fold,
            n_jobs=N_JOBS,
        )
        model.fit(
            X.iloc[tr_idx],
            y[tr_idx],
            eval_set=[(X.iloc[va_idx], y[va_idx])],
            verbose=250,
        )
        pred_va = model.predict_proba(X.iloc[va_idx])[:, 1]
        pred_test = model.predict_proba(X_test)[:, 1]
        oof[va_idx] = pred_va.astype("float32")
        test_pred += pred_test / N_SPLITS
        auc = auc_score(y[va_idx], pred_va)
        fold_aucs.append(auc)
        best_iter = getattr(model, "best_iteration", params.get("n_estimators", np.nan))
        best_iters.append(best_iter if best_iter is not None else params.get("n_estimators", np.nan))
        print(f"{name} fold={fold} auc={auc:.6f} best_iter={best_iters[-1]}")
        del model
        gc.collect()

    mean_auc = auc_score(y, oof)
    print(f"\n{name} OOF AUC={mean_auc:.6f}")
    save_pred_files(oof, test_pred, name)
    make_submission(test_pred, name)
    model_oof[name] = oof
    model_test[name] = test_pred
    model_meta.append({
        "model": name,
        "family": "xgboost",
        "feature_set": feature_set_name,
        "n_features": len(cols),
        "oof_auc": mean_auc,
        "fold_auc_mean": float(np.mean(fold_aucs)),
        "fold_auc_std": float(np.std(fold_aucs)),
        "mean_best_iter": float(np.nanmean(best_iters)),
        "cv_test_pred": True,
        "fullfit_test_pred": False,
    })
    return oof, test_pred

if RUN_XGBOOST and HAS_XGB:
    xgb_params = dict(
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        n_estimators=1600,
        learning_rate=0.018,
        max_depth=5,
        min_child_weight=12,
        subsample=0.82,
        colsample_bytree=0.78,
        reg_alpha=0.2,
        reg_lambda=12.0,
        scale_pos_weight=scale_pos_weight,
        early_stopping_rounds=120,
        max_bin=256,
    )
    if "v2_peak_freq_te" in feature_sets:
        train_xgb_cv("xgb_peak_freq_te", "v2_peak_freq_te", xgb_params)
    if "v2_pca_svd_stack" in feature_sets:
        p = xgb_params.copy()
        p.update({"max_depth": 4, "colsample_bytree": 0.88, "reg_lambda": 18.0})
        train_xgb_cv("xgb_pca_svd_stack", "v2_pca_svd_stack", p)


XGBoost model: xgb_peak_freq_te
feature_set: v2_peak_freq_te n_features: 1361

xgb_peak_freq_te fold 0
[0]	validation_0-auc:0.63858
[250]	validation_0-auc:0.70287
[465]	validation_0-auc:0.70424
xgb_peak_freq_te fold=0 auc=0.704981 best_iter=345

xgb_peak_freq_te fold 1
[0]	validation_0-auc:0.63477
[250]	validation_0-auc:0.69118
[364]	validation_0-auc:0.69043
xgb_peak_freq_te fold=1 auc=0.691383 best_iter=244

xgb_peak_freq_te fold 2
[0]	validation_0-auc:0.65508
[250]	validation_0-auc:0.70491
[402]	validation_0-auc:0.70290
xgb_peak_freq_te fold=2 auc=0.705618 best_iter=282

xgb_peak_freq_te fold 3
[0]	validation_0-auc:0.63578
[250]	validation_0-auc:0.70464
[500]	validation_0-auc:0.70597
[581]	validation_0-auc:0.70496
xgb_peak_freq_te fold=3 auc=0.706934 best_iter=461

xgb_peak_freq_te fold 4
[0]	validation_0-auc:0.66348
[250]	validation_0-auc:0.70907
[500]	validation_0-auc:0.71395
[536]	validation_0-auc:0.71281
xgb_peak_freq_te fold=4 auc=0.714213 best_iter=416

xgb_peak_freq_te OOF AU

In [16]:
# =========================
# 15. CatBoost
# =========================

def train_catboost_cv(name, feature_set_name, params):
    if not HAS_CATBOOST or not RUN_CATBOOST:
        print("Skipping CatBoost.")
        return None, None

    cols = feature_sets[feature_set_name]
    print("\n" + "=" * 100)
    print("CatBoost model:", name)
    print("feature_set:", feature_set_name, "n_features:", len(cols))
    print("=" * 100)

    X = X_pool_train[cols]
    X_test = X_pool_test[cols]

    oof = np.zeros(len(y), dtype="float32")
    test_pred = np.zeros(len(X_test), dtype="float64")
    fold_aucs = []
    best_iters = []

    for fold, (tr_idx, va_idx) in enumerate(folds):
        print(f"\n{name} fold {fold}")

        model = CatBoostClassifier(
            **params,
            random_seed=RANDOM_STATE + fold,
            thread_count=N_JOBS,
            verbose=250,
        )

        train_pool = Pool(X.iloc[tr_idx], y[tr_idx])
        valid_pool = Pool(X.iloc[va_idx], y[va_idx])

        model.fit(
            train_pool,
            eval_set=valid_pool,
            use_best_model=True,
            early_stopping_rounds=180,
        )

        pred_va = model.predict_proba(valid_pool)[:, 1]
        pred_test = model.predict_proba(Pool(X_test))[:, 1]
        oof[va_idx] = pred_va.astype("float32")
        test_pred += pred_test / N_SPLITS

        auc = auc_score(y[va_idx], pred_va)
        fold_aucs.append(auc)
        best_iters.append(int(model.get_best_iteration() or params.get("iterations", 0)))
        print(f"{name} fold={fold} auc={auc:.6f} best_iter={best_iters[-1]}")

        del model, train_pool, valid_pool
        gc.collect()

    mean_auc = auc_score(y, oof)
    print(f"\n{name} OOF AUC={mean_auc:.6f}")

    save_pred_files(oof, test_pred, name)
    make_submission(test_pred, name)

    model_oof[name] = oof
    model_test[name] = test_pred
    model_meta.append({
        "model": name,
        "family": "catboost",
        "feature_set": feature_set_name,
        "n_features": len(cols),
        "oof_auc": mean_auc,
        "fold_auc_mean": float(np.mean(fold_aucs)),
        "fold_auc_std": float(np.std(fold_aucs)),
        "mean_best_iter": float(np.mean(best_iters)),
        "cv_test_pred": True,
        "fullfit_test_pred": False,
    })
    return oof, test_pred

if RUN_CATBOOST and HAS_CATBOOST:
    cat_params = dict(
        loss_function="Logloss",
        eval_metric="AUC",
        iterations=3500,
        learning_rate=0.025,
        depth=6,
        l2_leaf_reg=18.0,
        random_strength=0.8,
        bagging_temperature=0.6,
        border_count=128,
        auto_class_weights="Balanced",
        allow_writing_files=False,
        task_type="CPU",
    )
    if "v2_peak_freq_te" in feature_sets:
        train_catboost_cv("catboost_peak_freq_te", "v2_peak_freq_te", cat_params)

    if "v2_linear_stack_compact" in feature_sets:
        cat_params2 = cat_params.copy()
        cat_params2.update({"depth": 5, "learning_rate": 0.03, "l2_leaf_reg": 25.0})
        train_catboost_cv("catboost_linear_stack_compact", "v2_linear_stack_compact", cat_params2)


CatBoost model: catboost_peak_freq_te
feature_set: v2_peak_freq_te n_features: 1361

catboost_peak_freq_te fold 0
0:	test: 0.6626036	best: 0.6626036 (0)	total: 121ms	remaining: 7m 3s
250:	test: 0.7000700	best: 0.7001091 (247)	total: 13.4s	remaining: 2m 53s
500:	test: 0.7026763	best: 0.7034916 (466)	total: 26.6s	remaining: 2m 39s
750:	test: 0.7029605	best: 0.7042019 (699)	total: 39.8s	remaining: 2m 25s
Stopped by overfitting detector  (180 iterations wait)

bestTest = 0.7042019138
bestIteration = 699

Shrink model to first 700 iterations.
catboost_peak_freq_te fold=0 auc=0.704202 best_iter=699

catboost_peak_freq_te fold 1
0:	test: 0.6403400	best: 0.6403400 (0)	total: 51.7ms	remaining: 3m
250:	test: 0.6999558	best: 0.6999558 (250)	total: 13.7s	remaining: 2m 56s
500:	test: 0.7009425	best: 0.7023085 (441)	total: 27.2s	remaining: 2m 42s
Stopped by overfitting detector  (180 iterations wait)

bestTest = 0.7023084508
bestIteration = 441

Shrink model to first 442 iterations.
catboost_peak_f

## 9. Сравнение моделей

Смотрим OOF AUC. Не надо автоматически сабмитить только top-1: sometimes модель с чуть меньшим CV хорошо дополняет ансамбль.

In [17]:
# =========================
# 16. Model leaderboard
# =========================

model_results = pd.DataFrame(model_meta)
model_results_path = OUT_DIR / "strong_v2_model_results.csv"
model_results.to_csv(model_results_path, index=False)
print("Saved:", model_results_path)

if len(model_results):
    display(model_results.sort_values("oof_auc", ascending=False))
else:
    print("No models trained?")

Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/model_outputs_v2/strong_v2_model_results.csv


,model,family,feature_set,n_features,oof_auc,fold_auc_mean,fold_auc_std,mean_best_iter,cv_test_pred,fullfit_test_pred
28,catboost_linear_stack_compact,catboost,v2_linear_stack_compact,1415,0.709989,0.713649,0.008691,200.8,True,False
20,lgb_optuna_best,lgbm,v2_peak_freq_te,1361,0.709474,0.710782,0.009183,998.0,True,False
16,lgb_peak_goss,lgbm,v2_peak_freq_te,1361,0.707886,0.708487,0.007706,503.2,True,False
18,lgb_peak_noise_downweight,lgbm,v2_peak_freq_te,1361,0.707554,0.710185,0.008861,598.2,True,False
0,lgb_peak_regularized,lgbm,v2_peak_freq_te,1361,0.706058,0.707801,0.009034,478.4,True,False
27,catboost_peak_freq_te,catboost,v2_peak_freq_te,1361,0.705806,0.708539,0.004589,481.2,True,False
25,xgb_peak_freq_te,xgboost,v2_peak_freq_te,1361,0.704309,0.704626,0.007399,349.6,True,False
23,histgb_v2_pca_svd_stack,sklearn,v2_pca_svd_stack,1165,0.703634,0.703963,0.006293,NaN,True,False
10,lgb_nodrift_top700,lgbm,v2_nodrift_top700_engineered,1869,0.703342,0.703757,0.005095,322.8,True,False
6,lgb_top500_engineered,lgbm,top500_plus_engineered,4589,0.702909,0.704239,0.007314,381.4,True,False


## 10. Ансамбли

Делаем несколько способов:

1. Equal probability blend.
2. Equal rank blend.
3. AUC-weighted blend.
4. Random-search weighted blend по OOF.
5. Logistic stacker на OOF-предсказаниях.
6. Rank-logistic stacker: то же самое, но на рангах моделей.

Сабмитить первым я бы пробовал:
- `random_weighted_blend`;
- `auc_weighted_blend`;
- `logistic_stacker`;
- лучший одиночный LightGBM.

In [18]:
# =========================
# 17. Blending / stacking
# =========================

assert len(model_oof) > 0, "No OOF predictions available."

names = list(model_oof.keys())
oof_matrix = np.column_stack([model_oof[n] for n in names])
test_matrix = np.column_stack([model_test[n] for n in names])

individual_auc = np.array([auc_score(y, model_oof[n]) for n in names])
indiv = pd.DataFrame({"model": names, "oof_auc": individual_auc}).sort_values("oof_auc", ascending=False)
display(indiv)

# Use top models for blend search.
top_names = indiv.head(min(BLEND_TOP_N_MODELS, len(indiv)))["model"].tolist()
top_idx = [names.index(n) for n in top_names]
oof_top = oof_matrix[:, top_idx]
test_top = test_matrix[:, top_idx]
auc_top = individual_auc[top_idx]

print("Top models for blending:")
for n, a in zip(top_names, auc_top):
    print(f"{n:40s} {a:.6f}")

blend_rows = []

def register_blend(name, oof_pred, test_pred, weights=None, model_names=None):
    auc = auc_score(y, oof_pred)
    print(f"{name}: OOF AUC={auc:.6f}")
    save_pred_files(oof_pred, test_pred, name)
    make_submission(test_pred, name)
    blend_rows.append({
        "blend": name,
        "oof_auc": auc,
        "n_models": len(model_names) if model_names is not None else np.nan,
        "models": "|".join(model_names) if model_names is not None else "",
        "weights": json.dumps([float(w) for w in weights]) if weights is not None else "",
    })
    return auc

# Equal probability
w_equal = np.ones(oof_top.shape[1]) / oof_top.shape[1]
register_blend(
    "blend_equal_probability_top",
    prob_blend(oof_top, w_equal),
    prob_blend(test_top, w_equal),
    w_equal,
    top_names,
)

# Equal rank
register_blend(
    "blend_equal_rank_top",
    rank_blend(oof_top, w_equal),
    rank_blend(test_top, w_equal),
    w_equal,
    top_names,
)

# AUC weighted: only excess over 0.5 matters.
w_auc = np.maximum(auc_top - 0.5, 1e-6)
w_auc = w_auc / w_auc.sum()
register_blend(
    "blend_auc_weighted_probability",
    prob_blend(oof_top, w_auc),
    prob_blend(test_top, w_auc),
    w_auc,
    top_names,
)

register_blend(
    "blend_auc_weighted_rank",
    rank_blend(oof_top, w_auc),
    rank_blend(test_top, w_auc),
    w_auc,
    top_names,
)

# Manual conservative: stronger weights to top models, small weights to diverse weaker models.
manual = np.array([max(0.02, (len(top_names) - i)) for i in range(len(top_names))], dtype=float)
manual = manual / manual.sum()
register_blend(
    "blend_manual_descending_probability",
    prob_blend(oof_top, manual),
    prob_blend(test_top, manual),
    manual,
    top_names,
)

# Random-search Dirichlet blend.
rng = np.random.default_rng(RANDOM_STATE)
best_auc = -1
best_w = None
best_kind = None
history = []

alpha_base = np.maximum((auc_top - 0.5) / max(1e-9, (auc_top - 0.5).mean()), 0.25)

for t in range(RANDOM_BLEND_TRIALS):
    # mix uniform and AUC-biased priors
    if t % 2 == 0:
        alpha = np.ones(len(top_names))
    else:
        alpha = alpha_base
    w = rng.dirichlet(alpha)
    pred = prob_blend(oof_top, w)
    auc = auc_score(y, pred)
    if auc > best_auc:
        best_auc = auc
        best_w = w.copy()
        best_kind = "prob"
    if t % 4 == 0:
        pred_r = rank_blend(oof_top, w)
        auc_r = auc_score(y, pred_r)
        if auc_r > best_auc:
            best_auc = auc_r
            best_w = w.copy()
            best_kind = "rank"
    if t % 1000 == 0:
        history.append((t, best_auc, best_kind))

print("Random blend best:", best_auc, best_kind)
for t, a, k in history[-10:]:
    print(t, a, k)

if best_kind == "prob":
    oof_best = prob_blend(oof_top, best_w)
    test_best = prob_blend(test_top, best_w)
else:
    oof_best = rank_blend(oof_top, best_w)
    test_best = rank_blend(test_top, best_w)

register_blend(
    f"blend_random_search_{best_kind}",
    oof_best,
    test_best,
    best_w,
    top_names,
)

pd.DataFrame({"model": top_names, "weight": best_w}).sort_values("weight", ascending=False).to_csv(OUT_DIR / "best_random_blend_weights.csv", index=False)
display(pd.DataFrame({"model": top_names, "weight": best_w, "auc": auc_top}).sort_values("weight", ascending=False))

# Logistic stacking on OOF predictions.
stack_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
stack_oof = np.zeros(len(y), dtype=float)
stack_test = np.zeros(len(test_index), dtype=float)

for fold, (tr, va) in enumerate(stack_cv.split(oof_top, y)):
    stacker = LogisticRegression(
        C=0.35,
        penalty="l2",
        solver="lbfgs",
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE + fold,
    )
    stacker.fit(oof_top[tr], y[tr])
    stack_oof[va] = stacker.predict_proba(oof_top[va])[:, 1]
    stack_test += stacker.predict_proba(test_top)[:, 1] / stack_cv.n_splits

register_blend(
    "blend_logistic_stacker_probability",
    stack_oof,
    stack_test,
    None,
    top_names,
)

# Rank logistic stacker.
oof_rank_top = np.column_stack([rank01(oof_top[:, i]) for i in range(oof_top.shape[1])])
test_rank_top = np.column_stack([rank01(test_top[:, i]) for i in range(test_top.shape[1])])

stack_oof_r = np.zeros(len(y), dtype=float)
stack_test_r = np.zeros(len(test_index), dtype=float)

for fold, (tr, va) in enumerate(stack_cv.split(oof_rank_top, y)):
    stacker = LogisticRegression(
        C=0.35,
        penalty="l2",
        solver="lbfgs",
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE + 100 + fold,
    )
    stacker.fit(oof_rank_top[tr], y[tr])
    stack_oof_r[va] = stacker.predict_proba(oof_rank_top[va])[:, 1]
    stack_test_r += stacker.predict_proba(test_rank_top)[:, 1] / stack_cv.n_splits

register_blend(
    "blend_logistic_stacker_rank",
    stack_oof_r,
    stack_test_r,
    None,
    top_names,
)

blend_df = pd.DataFrame(blend_rows).sort_values("oof_auc", ascending=False)
blend_df.to_csv(OUT_DIR / "strong_v2_blend_results.csv", index=False)
display(blend_df)

,model,oof_auc
17,catboost_linear_stack_compact,0.709989
10,lgb_optuna_best,0.709474
8,lgb_peak_goss,0.707886
9,lgb_peak_noise_downweight,0.707554
0,lgb_peak_regularized,0.706058
16,catboost_peak_freq_te,0.705806
14,xgb_peak_freq_te,0.704309
12,histgb_v2_pca_svd_stack,0.703634
5,lgb_nodrift_top700,0.703342
3,lgb_top500_engineered,0.702909


Top models for blending:
catboost_linear_stack_compact            0.709989
lgb_optuna_best                          0.709474
lgb_peak_goss                            0.707886
lgb_peak_noise_downweight                0.707554
lgb_peak_regularized                     0.706058
catboost_peak_freq_te                    0.705806
xgb_peak_freq_te                         0.704309
histgb_v2_pca_svd_stack                  0.703634
lgb_nodrift_top700                       0.703342
lgb_top500_engineered                    0.702909
lgb_peak_dart                            0.700699
xgb_pca_svd_stack                        0.699434
blend_equal_probability_top: OOF AUC=0.715456
Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/model_outputs_v2/oof_blend_equal_probability_top.csv
Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/model_outputs_v2/test_pred_blend_equal_probability_top.csv
Saved submission: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_ha

,model,weight,auc
0,catboost_linear_stack_compact,0.228893,0.709989
1,lgb_optuna_best,0.194922,0.709474
10,lgb_peak_dart,0.156533,0.700699
5,catboost_peak_freq_te,0.131876,0.705806
3,lgb_peak_noise_downweight,0.096622,0.707554
7,histgb_v2_pca_svd_stack,0.037050,0.703634
2,lgb_peak_goss,0.036220,0.707886
9,lgb_top500_engineered,0.035132,0.702909
6,xgb_peak_freq_te,0.029541,0.704309
4,lgb_peak_regularized,0.027028,0.706058


/Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and

blend_logistic_stacker_probability: OOF AUC=0.716513
Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/model_outputs_v2/oof_blend_logistic_stacker_probability.csv
Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/model_outputs_v2/test_pred_blend_logistic_stacker_probability.csv
Saved submission: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/submissions_v2/submission_blend_logistic_stacker_probability.csv


/Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and

blend_logistic_stacker_rank: OOF AUC=0.715784
Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/model_outputs_v2/oof_blend_logistic_stacker_rank.csv
Saved: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/model_outputs_v2/test_pred_blend_logistic_stacker_rank.csv
Saved submission: /Users/pinta/hse/mlproject/ml-project-adaai_VK_predict_hackaton/submissions_v2/submission_blend_logistic_stacker_rank.csv


,blend,oof_auc,n_models,models,weights
5,blend_random_search_prob,0.717250,12,catboost_linear_stack_compact|lgb_optuna_best|...,"[0.22889268454940281, 0.1949218591079289, 0.03..."
6,blend_logistic_stacker_probability,0.716513,12,catboost_linear_stack_compact|lgb_optuna_best|...,
7,blend_logistic_stacker_rank,0.715784,12,catboost_linear_stack_compact|lgb_optuna_best|...,
4,blend_manual_descending_probability,0.715775,12,catboost_linear_stack_compact|lgb_optuna_best|...,"[0.15384615384615385, 0.14102564102564102, 0.1..."
2,blend_auc_weighted_probability,0.715472,12,catboost_linear_stack_compact|lgb_optuna_best|...,"[0.0853235982910492, 0.08511419359114845, 0.08..."
0,blend_equal_probability_top,0.715456,12,catboost_linear_stack_compact|lgb_optuna_best|...,"[0.08333333333333333, 0.08333333333333333, 0.0..."
3,blend_auc_weighted_rank,0.715239,12,catboost_linear_stack_compact|lgb_optuna_best|...,"[0.0853235982910492, 0.08511419359114845, 0.08..."
1,blend_equal_rank_top,0.715223,12,catboost_linear_stack_compact|lgb_optuna_best|...,"[0.08333333333333333, 0.08333333333333333, 0.0..."


## 11. Optional postprocess по exact train/test overlap

Удаляем повторы

In [19]:
# =========================
# 18. Exact overlap postprocess
# =========================

overlap_path = PREP / "test_train_exact_overlap.csv"

def try_overlap_postprocess(base_submission_path: Path):
    if not overlap_path.exists():
        print("No overlap file:", overlap_path)
        return

    ov = pd.read_csv(overlap_path)
    print("Overlap file columns:", ov.columns.tolist(), "shape:", ov.shape)

    # Heuristic column detection
    test_col_candidates = [c for c in ov.columns if "test" in c.lower() and "index" in c.lower()] + [c for c in ov.columns if c.lower() in ("index", "test_index")]
    target_col_candidates = [c for c in ov.columns if c.lower() in ("target", "train_target", "target_mean", "mean_target")]

    if not test_col_candidates or not target_col_candidates:
        print("Could not infer overlap columns; skipping postprocess.")
        return

    test_col = test_col_candidates[0]
    target_col = target_col_candidates[0]

    sub = pd.read_csv(base_submission_path)
    score_col = "score" if "score" in sub.columns else sub.columns[-1]

    # If multiple train rows match one test row, use max/mean target depending on available table.
    ov_group = ov.groupby(test_col)[target_col].mean().reset_index()
    mapper = dict(zip(ov_group[test_col], ov_group[target_col]))

    # Conservative: if known/mean target is 1, push high; if 0, leave model alone.
    cons = sub.copy()
    m = cons["index"].map(mapper)
    pos_mask = m.fillna(0) > 0.5
    high_value = max(float(cons[score_col].max()), float(cons[score_col].quantile(0.9999)))
    cons.loc[pos_mask, score_col] = high_value
    out1 = base_submission_path.with_name(base_submission_path.stem + "_dup_conservative.csv")
    cons.to_csv(out1, index=False)
    print("Saved conservative overlap postprocess:", out1, "changed:", int(pos_mask.sum()))

    # Aggressive: use target mean to push both positives and negatives.
    aggr = sub.copy()
    known_mask = m.notna()
    if known_mask.any():
        lo = float(aggr[score_col].quantile(0.0001))
        hi = float(aggr[score_col].quantile(0.9999))
        aggr.loc[known_mask, score_col] = lo + m[known_mask].astype(float) * (hi - lo)
    out2 = base_submission_path.with_name(base_submission_path.stem + "_dup_aggressive.csv")
    aggr.to_csv(out2, index=False)
    print("Saved aggressive overlap postprocess:", out2, "changed:", int(known_mask.sum()))

# Apply to best few blend submissions
if len(blend_rows):
    best_blends = pd.DataFrame(blend_rows).sort_values("oof_auc", ascending=False)["blend"].head(5).tolist()
    for b in best_blends:
        path = SUB_DIR / f"submission_{safe_name(b)}.csv"
        if path.exists():
            try_overlap_postprocess(path)
else:
    print("No blend rows to postprocess.")

Overlap file columns: ['index', 'feature_hash', 'known_target_mean', 'train_count', 'train_positive_count'] shape: (8128, 5)
Could not infer overlap columns; skipping postprocess.
Overlap file columns: ['index', 'feature_hash', 'known_target_mean', 'train_count', 'train_positive_count'] shape: (8128, 5)
Could not infer overlap columns; skipping postprocess.
Overlap file columns: ['index', 'feature_hash', 'known_target_mean', 'train_count', 'train_positive_count'] shape: (8128, 5)
Could not infer overlap columns; skipping postprocess.
Overlap file columns: ['index', 'feature_hash', 'known_target_mean', 'train_count', 'train_positive_count'] shape: (8128, 5)
Could not infer overlap columns; skipping postprocess.
Overlap file columns: ['index', 'feature_hash', 'known_target_mean', 'train_count', 'train_positive_count'] shape: (8128, 5)
Could not infer overlap columns; skipping postprocess.
